# SSL-3: DINO — Self-Distillation with No Labels
**PM25Vision | Tasks 4–** | Backbone: EfficientNet-B0 | Multi-crop | Epochs: 30

In [1]:
# Cell 0 — GPU Reset
import gc, torch, os, warnings
warnings.filterwarnings("ignore")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.ipc_collect(); torch.cuda.synchronize()
    free  = torch.cuda.mem_get_info()[0]/1e9
    total = torch.cuda.mem_get_info()[1]/1e9
    print(f"GPU: {total:.1f} GB total | {total-free:.2f} GB used | {free:.2f} GB free")
else:
    print("No GPU — running on CPU.")


GPU: 15.6 GB total | 0.11 GB used | 15.53 GB free


In [2]:
# Cell 1 — Install
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q",
    "timm","thop","scikit-learn","umap-learn",
    "matplotlib","seaborn","scipy"], check=True)
print("Installs done.")


Installs done.


In [3]:
# Cell 2 — Imports
import os, gc, time, random, warnings, copy
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import timm
from thop import profile as thop_profile

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve, classification_report)
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from scipy import stats
import umap as umap_lib

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")


2026-04-18 18:09:36.317665: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776535776.534715      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776535776.606179      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776535777.124408      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776535777.124450      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776535777.124453      23 computation_placer.cc:177] computation placer alr

Device: cuda | PyTorch: 2.10.0+cu128


In [4]:
# Cell 3 — Config
SSL_NAME        = "DINO"
BACKBONE        = "efficientnet_b0"
BATCH_SIZE      = 16   # multi-crop: 2 global + 4 local = 6 views per image
PRETRAIN_EPOCHS = 30
TEACHER_TEMP    = 0.04
STUDENT_TEMP    = 0.1
CENTER_MOMENTUM = 0.9
EMA_MOMENTUM    = 0.996
IMG_SIZE        = 224
N_LOCAL         = 4
print(f"SSL: {SSL_NAME} | backbone: {BACKBONE} | batch: {BATCH_SIZE}")
print(f"teacher_temp={TEACHER_TEMP} | student_temp={STUDENT_TEMP} | ema={EMA_MOMENTUM}")


SSL: DINO | backbone: efficientnet_b0 | batch: 16
teacher_temp=0.04 | student_temp=0.1 | ema=0.996


In [5]:
# Cell 4 — Data loading
TRAIN_CSV = "/kaggle/input/datasets/deadcardassian/pm25vision/train/metadata.csv"
TEST_CSV  = "/kaggle/input/datasets/deadcardassian/pm25vision/test/metadata.csv"
TRAIN_IMG = "/kaggle/input/datasets/deadcardassian/pm25vision/train/images"
TEST_IMG  = "/kaggle/input/datasets/deadcardassian/pm25vision/test/images"

train_df = pd.read_csv(TRAIN_CSV); test_df = pd.read_csv(TEST_CSV)

def detect_cols(df):
    fc = next((c for c in df.columns if df[c].astype(str)
               .str.contains(r"\.(jpg|jpeg|png)", case=False, regex=True).any()), df.columns[0])
    lc = next((c for c in df.columns if c.lower() in
               ("bin","class","label","aqi","category","pm25_bin","aqi_bin","level")), df.columns[-1])
    return fc, lc

tr_f, tr_l = detect_cols(train_df); te_f, te_l = detect_cols(test_df)
all_labels  = sorted(pd.concat([train_df[tr_l], test_df[te_l]]).unique())
label_map   = {orig:i for i,orig in enumerate(all_labels)}
train_df["_label"] = train_df[tr_l].map(label_map)
test_df["_label"]  = test_df[te_l].map(label_map)
NUM_CLASSES = len(all_labels); CLASS_NAMES = [str(k) for k in all_labels]
full_df = pd.concat([train_df, test_df], ignore_index=True)
n_orig  = len(train_df)
full_df["_img_dir"] = [TRAIN_IMG if i < n_orig else TEST_IMG for i in range(len(full_df))]

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class PM25DS(Dataset):
    def __init__(self, df, fc, tf):
        self.df=df.reset_index(drop=True); self.fc=fc; self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row=self.df.iloc[i]
        img=Image.open(os.path.join(row["_img_dir"],os.path.basename(str(row[self.fc])))).convert("RGB")
        return self.tf(img), int(row["_label"])

print(f"Pool: {len(full_df)} images | {NUM_CLASSES} classes: {CLASS_NAMES}")


Pool: 11219 images | 6 classes: ['0–50', '101–150', '151–200', '201–300', '301–600', '51–100']


In [6]:
# Cell 5 — Augmentation recipe (DINO multi-crop)
# Global View 1: Heavy blur (p=1.0) — coarse haze structure
# Global View 2: Light blur + Solarize — different photometric distortion
# Local Views: small crops (96x96), medium blur — forces global prediction from local patch
color_jitter = transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)
normalize    = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

global_tf1 = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([color_jitter], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomApply([transforms.GaussianBlur(23, sigma=(0.1, 2.0))], p=1.0),
    transforms.ToTensor(), normalize
])
global_tf2 = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([color_jitter], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomApply([transforms.GaussianBlur(23, sigma=(0.1, 2.0))], p=0.1),
    transforms.RandomSolarize(128, p=0.2),
    transforms.ToTensor(), normalize
])
local_tf = transforms.Compose([
    transforms.RandomResizedCrop(96, scale=(0.05, 0.4)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([color_jitter], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomApply([transforms.GaussianBlur(9, sigma=(0.1, 2.0))], p=0.5),
    transforms.ToTensor(), normalize
])

class DINODataset(Dataset):
    def __init__(self, df, fc):
        self.df=df.reset_index(drop=True); self.fc=fc
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row=self.df.iloc[i]
        img=Image.open(os.path.join(row["_img_dir"],os.path.basename(str(row[self.fc])))).convert("RGB")
        views = [global_tf1(img), global_tf2(img)] + [local_tf(img) for _ in range(N_LOCAL)]
        return views, int(row["_label"])

def dino_collate(batch):
    views_list, labels = zip(*batch)
    n_views = len(views_list[0])
    stacked = [torch.stack([v[i] for v in views_list]) for i in range(n_views)]
    return stacked, torch.tensor(labels)

pretrain_ds  = DINODataset(full_df, tr_f)
pretrain_ldr = DataLoader(pretrain_ds, BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True,
                          collate_fn=dino_collate)
print(f"Multi-crop: 2 global (224x224) + {N_LOCAL} local (96x96)")
print(f"Pretrain loader: {len(pretrain_ldr)} batches/epoch")


Multi-crop: 2 global (224x224) + 4 local (96x96)
Pretrain loader: 701 batches/epoch


In [7]:
# Cell 6 — DINO model definition
class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim, bottleneck_dim=256, n_layers=3):
        super().__init__()
        layers = []
        for i in range(n_layers - 1):
            layers += [nn.Linear(in_dim if i==0 else bottleneck_dim, bottleneck_dim),
                       nn.BatchNorm1d(bottleneck_dim), nn.GELU()]
        layers.append(nn.Linear(bottleneck_dim, out_dim))
        self.mlp = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return F.normalize(self.mlp(x), dim=-1, p=2)

class DINO(nn.Module):
    def __init__(self, backbone=BACKBONE, teacher_temp=TEACHER_TEMP,
                 student_temp=STUDENT_TEMP, center_mom=CENTER_MOMENTUM):
        super().__init__()
        self.student_enc  = timm.create_model(backbone, pretrained=True, num_classes=0)
        num_ftrs = self.student_enc.num_features
        self.student_head = DINOHead(num_ftrs, 65536)
        self.teacher_enc  = timm.create_model(backbone, pretrained=True, num_classes=0)
        self.teacher_head = DINOHead(num_ftrs, 65536)
        self.teacher_enc.load_state_dict(self.student_enc.state_dict())
        self.teacher_head.load_state_dict(self.student_head.state_dict())
        for p in list(self.teacher_enc.parameters()) + list(self.teacher_head.parameters()):
            p.requires_grad = False
        self.register_buffer("center", torch.zeros(1, 65536))
        self.tau_t, self.tau_s, self.center_mom = teacher_temp, student_temp, center_mom

    @torch.no_grad()
    def ema_update(self, momentum=EMA_MOMENTUM):
        for s,t in zip(self.student_enc.parameters(), self.teacher_enc.parameters()):
            t.data.mul_(momentum).add_(s.data, alpha=1-momentum)
        for s,t in zip(self.student_head.parameters(), self.teacher_head.parameters()):
            t.data.mul_(momentum).add_(s.data, alpha=1-momentum)

    def forward(self, views):
        with torch.no_grad():
            t_out = torch.cat([self.teacher_head(self.teacher_enc(views[0])),
                               self.teacher_head(self.teacher_enc(views[1]))])
            t_probs = F.softmax((t_out - self.center) / self.tau_t, dim=-1)
        s_out = torch.cat([self.student_head(self.student_enc(v)) for v in views])
        s_log = F.log_softmax(s_out / self.tau_s, dim=-1)
        loss = 0; n_s = len(views); n_t = 2
        for i in range(n_t):
            for j in range(n_s):
                if i == j: continue
                loss -= (t_probs[i] * s_log[j]).sum(dim=-1).mean()
        with torch.no_grad():
            self.center.mul_(self.center_mom).add_(t_out.mean(dim=0), alpha=1-self.center_mom)
        return loss / (n_t * (n_s - 1))

model = DINO().to(DEVICE)

# GFLOPs computation (fixed — was missing in original NB3)
dummy = torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE)
with torch.no_grad():
    macs,_ = thop_profile(model.student_enc, inputs=(dummy,), verbose=False)
gflops = macs/1e9
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
print(f"Params (trainable): {total_params:.1f}M | GFLOPs: {gflops:.3f}")
torch.cuda.empty_cache()
print("DINO model initialized.")


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Params (trainable): 21.2M | GFLOPs: 0.385
DINO model initialized.


In [8]:
# Cell 7 — Pretraining (Task 5.1)
opt = optim.AdamW([
    {"params": list(model.student_enc.parameters()),  "lr": 5e-5},
    {"params": list(model.student_head.parameters()), "lr": 3e-4},
], weight_decay=0.04)
sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PRETRAIN_EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

losses = []; pretrain_time_start = time.time()
for ep in range(1, PRETRAIN_EPOCHS+1):
    model.train(); ep_l=0; nb=0
    for views, _ in pretrain_ldr:
        views = [v.to(DEVICE) for v in views]
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            loss = model(views)
        if torch.isnan(loss):
            print(f"  NaN loss at ep {ep} — skipping batch"); continue
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=3.0)
        scaler.step(opt); scaler.update()
        model.ema_update(); ep_l+=loss.item(); nb+=1
    if nb == 0: losses.append(float("nan")); continue
    sch.step(); avg=ep_l/nb; losses.append(avg)
    if ep%5==0 or ep==1:
        mem = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
        print(f"Ep {ep:03d}/{PRETRAIN_EPOCHS} loss={avg:.4f} gpu={mem:.2f}GB")

pretrain_time = time.time()-pretrain_time_start
torch.cuda.empty_cache()
torch.save(model.state_dict(), "dino_encoder.pt")
print(f"Saved: dino_encoder.pt | {pretrain_time:.0f}s ({pretrain_time/60:.1f} min)")

valid_losses = [l for l in losses if not (isinstance(l,float) and l!=l)]
plt.figure(figsize=(8,4))
plt.plot(valid_losses, lw=2); plt.xlabel("Epoch"); plt.ylabel("DINO Cross-Entropy")
plt.title("DINO Pretraining Loss"); plt.grid(True,alpha=0.5)
plt.tight_layout(); plt.savefig("pretrain_loss_DINO.png",dpi=150); plt.show()

encoder = model.student_enc
for p in encoder.parameters(): p.requires_grad_(False)
encoder.eval(); torch.cuda.empty_cache()
print("Student encoder frozen.")


Ep 001/30 loss=11.0906 gpu=0.47GB
Ep 005/30 loss=11.0904 gpu=0.47GB
Ep 010/30 loss=11.0904 gpu=0.47GB
Ep 015/30 loss=11.0904 gpu=0.47GB
Ep 020/30 loss=11.0901 gpu=0.47GB
Ep 025/30 loss=11.0885 gpu=0.47GB
Ep 030/30 loss=11.0882 gpu=0.47GB
Saved: dino_encoder.pt | 19452s (324.2 min)
Student encoder frozen.


In [9]:
# Cell 8 — Feature extraction
def extract_features(enc, loader):
    enc.eval(); feats, lbls = [], []
    with torch.no_grad():
        for x, y in loader:
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                feats.append(enc(x.to(DEVICE)).cpu().float().numpy())
            lbls.extend(y.numpy())
    return np.concatenate(feats), np.array(lbls)

tr_idx, te_idx = train_test_split(
    list(range(len(full_df))), test_size=0.20,
    stratify=full_df["_label"].tolist(), random_state=SEED)

train_ldr_ev = DataLoader(PM25DS(full_df.iloc[tr_idx], tr_f, eval_tf), 64, num_workers=2, pin_memory=True)
test_ldr_ev  = DataLoader(PM25DS(full_df.iloc[te_idx], tr_f, eval_tf), 64, num_workers=2, pin_memory=True)

print("Extracting frozen features...")
t0 = time.time()
X_tr, y_tr = extract_features(encoder, train_ldr_ev)
X_te, y_te = extract_features(encoder, test_ldr_ev)
feat_time = time.time()-t0
print(f"Train {X_tr.shape} | Test {X_te.shape} | {feat_time:.1f}s")

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)


Extracting frozen features...
Train (8975, 1280) | Test (2244, 1280) | 96.0s


In [10]:
# Cell 9 — Shallow heads (Task 5.2)
def eval_head(name, clf):
    t0=time.time(); clf.fit(X_tr_s, y_tr); tr_t=time.time()-t0
    t0=time.time(); preds=clf.predict(X_te_s); te_t=time.time()-t0
    acc=accuracy_score(y_te,preds)
    f1 =f1_score(y_te,preds,average="macro",zero_division=0)
    try:
        prob=clf.predict_proba(X_te_s)
        auc=roc_auc_score(y_te,prob,multi_class="ovr",average="macro")
    except: auc=float("nan")
    print(f"  {name:22s} acc={acc:.4f}  f1={f1:.4f}  auc={auc:.4f}  fit={tr_t:.1f}s")
    return {"head":name,"acc":acc,"f1":f1,"auc":auc,"train_t":round(tr_t,2),"test_t":round(te_t,3)}

print("── Shallow heads on frozen features ──")
heads = []
heads.append(eval_head("Linear Probe",    LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)))
heads.append(eval_head("MLP",             MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=SEED)))
heads.append(eval_head("SVM (RBF)",       SVC(kernel="rbf", C=10, probability=True, random_state=SEED)))
heads.append(eval_head("Decision Tree",   DecisionTreeClassifier(max_depth=12, random_state=SEED)))
heads.append(eval_head("Random Forest",   RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED)))
df_heads = pd.DataFrame(heads)
print(); print(df_heads[["head","acc","f1","auc"]].to_string(index=False))


── Shallow heads on frozen features ──
  Linear Probe           acc=0.4550  f1=0.4753  auc=0.7902  fit=36.7s
  MLP                    acc=0.5352  f1=0.5612  auc=0.8301  fit=25.2s
  SVM (RBF)              acc=0.5441  f1=0.5814  auc=0.8459  fit=281.4s
  Decision Tree          acc=0.3039  f1=0.2933  auc=0.5927  fit=15.7s
  Random Forest          acc=0.4091  f1=0.3791  auc=0.7562  fit=13.2s

         head      acc       f1      auc
 Linear Probe 0.454991 0.475295 0.790249
          MLP 0.535205 0.561216 0.830119
    SVM (RBF) 0.544118 0.581449 0.845947
Decision Tree 0.303922 0.293293 0.592690
Random Forest 0.409091 0.379099 0.756178


In [11]:
# Cell 10 — Full fine-tuning vs linear probe (Task 5.2 cont.)
print("── Full fine-tuning ──")
FINETUNE_EPOCHS = 20; LR_FT = 1e-4

class FTModel(nn.Module):
    def __init__(self, enc, feat_dim, n_cls):
        super().__init__(); self.enc=enc; self.head=nn.Linear(feat_dim, n_cls)
    def forward(self, x): return self.head(self.enc(x))

for p in encoder.parameters(): p.requires_grad_(True)
ft_model  = FTModel(encoder, X_tr.shape[1], NUM_CLASSES).to(DEVICE)
ft_opt    = optim.AdamW(ft_model.parameters(), lr=LR_FT, weight_decay=1e-4)
ft_sch    = optim.lr_scheduler.CosineAnnealingLR(ft_opt, T_max=FINETUNE_EPOCHS)
ft_crit   = nn.CrossEntropyLoss()
ft_scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
ft_aug    = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.2,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3,0.3,0.3,0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
ft_train_ldr = DataLoader(PM25DS(full_df.iloc[tr_idx], tr_f, ft_aug),
                          batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

ft_hist = {"loss":[],"acc":[]}
ft_wall_start = time.time()
for ep in range(1, FINETUNE_EPOCHS+1):
    ft_model.train(); ep_l=0; ep_c=0; ep_n=0
    for x,y in ft_train_ldr:
        x,y=x.to(DEVICE),y.to(DEVICE); ft_opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            out=ft_model(x); loss=ft_crit(out,y)
        ft_scaler.scale(loss).backward(); ft_scaler.step(ft_opt); ft_scaler.update()
        ep_l+=loss.item(); ep_c+=(out.argmax(1)==y).sum().item(); ep_n+=y.size(0)
    ft_sch.step()
    ft_hist["loss"].append(ep_l/len(ft_train_ldr)); ft_hist["acc"].append(ep_c/ep_n)
    if ep%5==0 or ep==1:
        print(f"  FT ep {ep:02d}/{FINETUNE_EPOCHS}  loss={ep_l/len(ft_train_ldr):.4f}  acc={ep_c/ep_n:.4f}")
ft_wall_time = time.time()-ft_wall_start

ft_model.eval(); ft_preds=[]; ft_probs=[]
with torch.no_grad():
    for x,y in test_ldr_ev:
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits=ft_model(x.to(DEVICE))
        ft_preds.extend(logits.argmax(1).cpu().numpy())
        ft_probs.extend(torch.softmax(logits,1).cpu().float().numpy())
ft_preds=np.array(ft_preds); ft_probs=np.array(ft_probs)
ft_acc=accuracy_score(y_te,ft_preds)
ft_f1=f1_score(y_te,ft_preds,average="macro",zero_division=0)
try:    ft_auc=roc_auc_score(y_te,ft_probs,multi_class="ovr",average="macro")
except: ft_auc=float("nan")
lp_acc=heads[0]["acc"]
print(f"  Linear Probe acc : {lp_acc:.4f}")
print(f"  Fine-Tuned  acc  : {ft_acc:.4f}  (Δ = {ft_acc-lp_acc:+.4f})")
print(f"  Fine-Tuned  F1   : {ft_f1:.4f} | AUC: {ft_auc:.4f}")

fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4))
a1.plot(ft_hist["loss"],lw=2); a1.set_title("Fine-tune Loss"); a1.set_xlabel("Epoch"); a1.grid(True,alpha=0.5)
a2.plot(ft_hist["acc"],lw=2,color="goldenrod"); a2.set_title("Fine-tune Train Acc"); a2.set_xlabel("Epoch"); a2.grid(True,alpha=0.5)
plt.suptitle(f"{SSL_NAME} Fine-tuning Curves"); plt.tight_layout()
plt.savefig(f"finetune_curves_{SSL_NAME}.png",dpi=150); plt.show()

for p in encoder.parameters(): p.requires_grad_(False)
encoder.eval(); torch.cuda.empty_cache()


── Full fine-tuning ──
  FT ep 01/20  loss=1.5683  acc=0.3392
  FT ep 05/20  loss=0.8560  acc=0.6497
  FT ep 10/20  loss=0.5739  acc=0.7756
  FT ep 15/20  loss=0.4264  acc=0.8399
  FT ep 20/20  loss=0.3635  acc=0.8728
  Linear Probe acc : 0.4550
  Fine-Tuned  acc  : 0.6836  (Δ = +0.2286)
  Fine-Tuned  F1   : 0.7204 | AUC: nan


In [12]:
# Cell 11 — k-NN & label efficiency (Task 5.4)
print("── k-NN in embedding space ──")
knn_res = []
for k in [1, 5, 20]:
    knn=KNeighborsClassifier(n_neighbors=k, metric="cosine", n_jobs=-1)
    knn.fit(X_tr_s, y_tr)
    acc_k=accuracy_score(y_te, knn.predict(X_te_s))
    print(f"  k={k:2d}: {acc_k:.4f}"); knn_res.append({"k":k,"acc":round(acc_k,4)})
df_knn = pd.DataFrame(knn_res)

print("── Label-efficiency curve ──")
le_res = []
for frac in [0.01, 0.05, 0.10, 0.25, 0.50]:
    n = max(NUM_CLASSES, int(frac*len(X_tr)))
    idx2, _ = train_test_split(range(len(X_tr)), train_size=n, stratify=y_tr, random_state=SEED)
    lr2=LogisticRegression(max_iter=500, C=1.0, random_state=SEED)
    lr2.fit(X_tr_s[idx2], y_tr[idx2])
    acc2=accuracy_score(y_te, lr2.predict(X_te_s))
    print(f"  {int(frac*100):3d}% ({n:4d} samples): {acc2:.4f}")
    le_res.append({"pct":int(frac*100),"n":n,"acc":round(acc2,4)})
df_le = pd.DataFrame(le_res)

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))
ax1.bar([r["head"] for r in heads],[r["acc"] for r in heads], edgecolor="k",
        color=plt.cm.tab10(np.linspace(0,0.5,len(heads))))
ax1.set_xticklabels([r["head"] for r in heads],rotation=25,ha="right",fontsize=9)
ax1.set_ylabel("Accuracy"); ax1.set_title(f"{SSL_NAME} — Frozen Feature Heads"); ax1.grid(axis="y",alpha=0.4)
ax2.plot(df_le["pct"], df_le["acc"], "o-", lw=2, ms=7)
ax2.set_xlabel("Labeled data (%)"); ax2.set_ylabel("Linear Probe Accuracy")
ax2.set_title(f"{SSL_NAME} — Label-Efficiency"); ax2.grid(True,alpha=0.4)
plt.tight_layout(); plt.savefig(f"heads_le_{SSL_NAME}.png",dpi=150); plt.show()


── k-NN in embedding space ──
  k= 1: 0.5798
  k= 5: 0.5463
  k=20: 0.5250
── Label-efficiency curve ──
    1% (  89 samples): 0.3017
    5% ( 448 samples): 0.3391
   10% ( 897 samples): 0.3596
   25% (2243 samples): 0.3730
   50% (4487 samples): 0.4113


In [13]:
# Cell 12 — Full metrics: per-class acc, confusion matrix, ROC (Task 5.4)
best_clf = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
best_clf.fit(X_tr_s, y_tr)
preds = best_clf.predict(X_te_s)
prob  = best_clf.predict_proba(X_te_s)

acc      = accuracy_score(y_te, preds)
prec_mac = precision_score(y_te, preds, average="macro", zero_division=0)
rec_mac  = recall_score(y_te, preds,    average="macro", zero_division=0)
f1_mac   = f1_score(y_te, preds,        average="macro", zero_division=0)
try:    roc_auc = roc_auc_score(y_te, prob, multi_class="ovr", average="macro")
except: roc_auc = float("nan")
cm = confusion_matrix(y_te, preds)
pca_acc = cm.diagonal() / cm.sum(axis=1)

print("="*58)
for lbl, val in [("Accuracy",acc),("Macro Precision",prec_mac),
                 ("Macro Recall",rec_mac),("Macro F1",f1_mac),("ROC-AUC",roc_auc)]:
    print(f"  {lbl:20s}: {val:.4f}")
print(f"  {'Pretrain time':20s}: {pretrain_time:.0f}s")
print(f"  {'Fine-tune time':20s}: {ft_wall_time:.0f}s")
print(f"  {'GFLOPs':20s}: {gflops:.3f}")
print("="*58)
print("Per-class accuracy:")
for n,pa in zip(CLASS_NAMES,pca_acc): print(f"  Class {n}: {pa:.4f}")
print(); print(classification_report(y_te, preds, target_names=CLASS_NAMES))

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(13,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1, linewidths=0.5)
ax1.set_title(f"{SSL_NAME} — Confusion Matrix (Linear Probe)")
ax1.set_xlabel("Predicted"); ax1.set_ylabel("True")
lbls_bin = label_binarize(y_te, classes=list(range(NUM_CLASSES)))
for i in range(NUM_CLASSES):
    try:
        fpr,tpr,_=roc_curve(lbls_bin[:,i],prob[:,i])
        auc_i=roc_auc_score(lbls_bin[:,i],prob[:,i])
        ax2.plot(fpr,tpr,lw=2,label=f"Class {CLASS_NAMES[i]} ({auc_i:.3f})")
    except: pass
ax2.plot([0,1],[0,1],"k--",lw=1); ax2.legend(fontsize=8); ax2.grid(True,alpha=0.4)
ax2.set_title(f"{SSL_NAME} — ROC Curves"); ax2.set_xlabel("FPR"); ax2.set_ylabel("TPR")
plt.tight_layout(); plt.savefig(f"metrics_{SSL_NAME}.png",dpi=150); plt.show()


  Accuracy            : 0.4550
  Macro Precision     : 0.4652
  Macro Recall        : 0.4883
  Macro F1            : 0.4753
  ROC-AUC             : 0.7902
  Pretrain time       : 19452s
  Fine-tune time      : 1826s
  GFLOPs              : 0.385
Per-class accuracy:
  Class 0–50: 0.4544
  Class 101–150: 0.4502
  Class 151–200: 0.5249
  Class 201–300: 0.6182
  Class 301–600: 0.5938
  Class 51–100: 0.2884

              precision    recall  f1-score   support

        0–50       0.45      0.45      0.45       482
     101–150       0.46      0.45      0.45       482
     151–200       0.52      0.52      0.52       482
     201–300       0.51      0.62      0.56       220
     301–600       0.52      0.59      0.56        96
      51–100       0.33      0.29      0.31       482

    accuracy                           0.45      2244
   macro avg       0.47      0.49      0.48      2244
weighted avg       0.45      0.45      0.45      2244



In [14]:
# Cell 13 — Embedding analysis: PCA, t-SNE, UMAP, Silhouette (Task 5.3)
N = min(2000, len(X_te))
idx_p = np.random.choice(len(X_te), N, replace=False)
Xp = X_te_s[idx_p]; yp = y_te[idx_p]
pal = plt.cm.tab10(np.linspace(0,0.6,NUM_CLASSES))

print("Running PCA..."); Xpca  = PCA(2, random_state=SEED).fit_transform(Xp)
print("Running t-SNE..."); Xtsne = TSNE(2, random_state=SEED, perplexity=40, n_iter=1000).fit_transform(Xp)
print("Running UMAP..."); Xumap = umap_lib.UMAP(n_components=2, random_state=SEED, n_neighbors=30).fit_transform(Xp)
sil = silhouette_score(Xp, yp)
print(f"Silhouette score: {sil:.4f}")

fig,axes = plt.subplots(1,3,figsize=(18,5))
for ax, X2, title in zip(axes,[Xpca,Xtsne,Xumap],["PCA","t-SNE","UMAP"]):
    for c in range(NUM_CLASSES):
        m = yp==c
        ax.scatter(X2[m,0],X2[m,1],s=8,alpha=0.6,color=pal[c],label=CLASS_NAMES[c])
    ax.set_title(f"{SSL_NAME} — {title}"); ax.legend(fontsize=7,markerscale=2)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f"Silhouette = {sil:.4f}  |  {SSL_NAME} frozen encoder", fontsize=12)
plt.tight_layout(); plt.savefig(f"embeddings_{SSL_NAME}.png",dpi=150); plt.show()


Running PCA...
Running t-SNE...
Running UMAP...
Silhouette score: -0.0305


In [15]:
# Cell 16 — Save complete summary CSV
summary = {
    "SSL":SSL_NAME, "Backbone":BACKBONE,
    "Accuracy":round(acc,4), "Macro Precision":round(prec_mac,4),
    "Macro Recall":round(rec_mac,4), "Macro F1":round(f1_mac,4),
    "Macro ROC-AUC":round(float(roc_auc),4), "Silhouette":round(sil,4),
    "FT Accuracy":round(ft_acc,4), "FT F1":round(ft_f1,4),
    "LP vs FT delta":round(ft_acc-acc,4),
    "kNN-1": df_knn[df_knn.k==1]["acc"].values[0],
    "kNN-5": df_knn[df_knn.k==5]["acc"].values[0],
    "kNN-20":df_knn[df_knn.k==20]["acc"].values[0],
    "Pretrain Time (s)":round(pretrain_time,1),
    "FineTune Time (s)":round(ft_wall_time,1),
    "GFLOPs":round(gflops,3),
}
pd.DataFrame([summary]).to_csv(f"ssl_summary_{SSL_NAME}.csv", index=False)
df_heads.to_csv(f"heads_{SSL_NAME}.csv", index=False)
df_le.to_csv(f"label_eff_{SSL_NAME}.csv", index=False)
print("All files saved:")
for k,v in summary.items(): print(f"  {k:25s}: {v}")


All files saved:
  SSL                      : DINO
  Backbone                 : efficientnet_b0
  Accuracy                 : 0.455
  Macro Precision          : 0.4652
  Macro Recall             : 0.4883
  Macro F1                 : 0.4753
  Macro ROC-AUC            : 0.7902
  Silhouette               : -0.030500000342726707
  FT Accuracy              : 0.6836
  FT F1                    : 0.7204
  LP vs FT delta           : 0.2286
  kNN-1                    : 0.5798
  kNN-5                    : 0.5463
  kNN-20                   : 0.525
  Pretrain Time (s)        : 19451.9
  FineTune Time (s)        : 1826.4
  GFLOPs                   : 0.385
